# 魔術方法 -- 讓你的類別更 Pythonic

## 學習目標

- 理解魔術方法（special methods / dunder methods）的作用
- 學會 `__str__` 和 `__repr__` 的差異
- 掌握比較運算子與算術運算子的覆寫
- 認識容器協定（`__len__`、`__getitem__`）

---

## 1. 什麼是魔術方法？

魔術方法是 Python 中以雙底線包圍的特殊方法（如 `__init__`），
它們讓你的類別可以和 Python 的內建語法互動：

```
len(obj)      # 呼叫 obj.__len__()
str(obj)      # 呼叫 obj.__str__()
obj[0]        # 呼叫 obj.__getitem__(0)
obj1 + obj2   # 呼叫 obj1.__add__(obj2)
```

你不會直接呼叫這些方法，而是透過 Python 的語法來觸發它們。

## 2. \_\_str\_\_ vs \_\_repr\_\_

這是最常用的兩個魔術方法：

In [ ]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __str__(self):
        """給人看的：print() 時使用"""
        return f"{self.name} - NT${self.price}"

    def __repr__(self):
        """給開發者看的：除錯時使用，應能重建物件"""
        return f"Product({self.name!r}, {self.price!r})"

In [ ]:
p = Product("咖啡", 120)

print(p)        # 咖啡 - NT$120        （呼叫 __str__）
print(repr(p))  # Product('咖啡', 120)  （呼叫 __repr__）
print([p])      # [Product('咖啡', 120)] （容器內用 __repr__）

**規則：**
- 只實作一個的話，選 `__repr__`（`str()` 找不到 `__str__` 時會退而求其次用 `__repr__`）
- `__repr__` 的輸出理想上應該能用 `eval()` 重建物件

## 3. 比較運算子

In [ ]:
from functools import total_ordering

@total_ordering  # 只要定義 __eq__ 和 __lt__，自動產生其餘比較方法
class Student:
    def __init__(self, name, grade):
        self.name = name
        self.grade = grade

    def __eq__(self, other):
        return self.grade == other.grade

    def __lt__(self, other):
        return self.grade < other.grade

    def __repr__(self):
        return f"Student({self.name!r}, {self.grade})"

In [ ]:
s1 = Student("小明", 85)
s2 = Student("小華", 92)

print(s1 < s2)    # True
print(s1 >= s2)   # False （由 @total_ordering 自動產生）
print(s1 == s2)   # False

# 因為定義了比較方法，可以直接排序
students = [Student("A", 78), Student("B", 95), Student("C", 82)]
print(sorted(students))  # [Student('A', 78), Student('C', 82), Student('B', 95)]

`@total_ordering` 很實用：你只需要定義 `__eq__` + 一個不等式（通常是 `__lt__`），
它就會自動幫你補上 `__le__`、`__gt__`、`__ge__`。

## 4. 算術運算子 -- Vector 範例

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        """v1 + v2"""
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        """v1 - v2"""
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):
        """v * 3（向量乘以純量）"""
        return Vector(self.x * scalar, self.y * scalar)

    def __abs__(self):
        """abs(v) 回傳向量長度"""
        return (self.x ** 2 + self.y ** 2) ** 0.5

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

In [ ]:
v1 = Vector(3, 4)
v2 = Vector(1, 2)

print(v1 + v2)    # Vector(4, 6)
print(v1 - v2)    # Vector(2, 2)
print(v1 * 3)     # Vector(9, 12)
print(abs(v1))    # 5.0

注意：這裡每個運算都回傳**新的 Vector**，不修改原物件（不可變性原則）。

## 5. 容器方法

讓你的類別可以像 list 或 dict 一樣使用：

In [ ]:
class Playlist:
    def __init__(self, name, songs=None):
        self.name = name
        self._songs = list(songs) if songs else []

    def __len__(self):
        """len(playlist)"""
        return len(self._songs)

    def __getitem__(self, index):
        """playlist[0] 或 playlist[1:3]"""
        return self._songs[index]

    def __contains__(self, song):
        """'song_name' in playlist"""
        return song in self._songs

    def __repr__(self):
        return f"Playlist({self.name!r}, {len(self)} songs)"

In [ ]:
pl = Playlist("我的最愛", ["晴天", "稻香", "七里香"])

print(len(pl))          # 3
print(pl[0])            # 晴天
print(pl[1:])           # ['稻香', '七里香']
print("稻香" in pl)     # True

# 因為有 __getitem__，自動支援 for 迴圈
for song in pl:
    print(song)

## 6. \_\_call\_\_ -- 讓物件可以被呼叫

In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, value):
        return value * self.factor

double = Multiplier(2)
print(double(5))    # 10
print(double(100))  # 200

`__call__` 讓物件的行為像函式，適合用在需要「帶狀態的函式」的場景。

## 7. 練習題

建立一個 `Money` 類別：
1. 有 `amount` 和 `currency` 屬性
2. 實作 `__str__`（顯示如 "NT$100"）和 `__repr__`
3. 實作 `__add__`（同幣別才能相加，不同幣別拋出 ValueError）
4. 實作 `__eq__` 和 `__lt__`

In [ ]:
# 在這裡寫你的練習程式碼


---

## 常用魔術方法參考表

| 方法 | 觸發方式 | 用途 |
|---|---|---|
| `__init__` | `MyClass()` | 初始化物件 |
| `__str__` | `str(obj)`, `print(obj)` | 人類可讀的字串 |
| `__repr__` | `repr(obj)` | 開發者用的字串 |
| `__eq__` | `obj1 == obj2` | 相等比較 |
| `__lt__` | `obj1 < obj2` | 小於比較 |
| `__add__` | `obj1 + obj2` | 加法運算 |
| `__sub__` | `obj1 - obj2` | 減法運算 |
| `__mul__` | `obj1 * obj2` | 乘法運算 |
| `__len__` | `len(obj)` | 長度 |
| `__getitem__` | `obj[key]` | 索引取值 |
| `__contains__` | `item in obj` | 成員檢查 |
| `__call__` | `obj()` | 把物件當函式呼叫 |
| `__abs__` | `abs(obj)` | 絕對值 |
| `__bool__` | `bool(obj)`, `if obj:` | 布林轉換 |

---

> 上一篇：[02_polymorphism.ipynb](02_polymorphism.ipynb) -- 多型與抽象類別
>
> 下一篇：[04_multiple_inheritance.ipynb](04_multiple_inheritance.ipynb) -- 多重繼承與 Mixin